![image.png](https://i.imgur.com/4fN73lZ.png)

This notebook has been inspired by [Q* Learning with FrozenLakev2.ipynb](https://colab.research.google.com/github/simoninithomas/Deep_reinforcement_learning_Course/blob/master/Q_Learning_with_FrozenLakev2.ipynb#scrollTo=Xr9nI6dcQM8I) and [Deep Reinforcement Learning Course](https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt) by Thomas Simonini

### Setup

We standardize on **Gymnasium** (the maintained successor to OpenAI Gym) plus `imageio` for rendering rollouts as GIFs.

In [ ]:
# Fast dependency install with uv (https://docs.astral.sh/uv).
# Bootstraps uv via pip, then installs into the current kernel/venv. If you already
# run from the project's .venv (created with `uv sync`), this is essentially a no-op.
import sys, os
%pip install -q uv
_target = "" if (sys.prefix != sys.base_prefix or os.environ.get("VIRTUAL_ENV")) else "--system"
!uv pip install -q {_target} --python "{sys.executable}" "gymnasium[toy-text]" imageio matplotlib

# Q-Learning

In this notebook, we will implement Q-Learning Reinforcement learning algorithm for Frozen Lake Environment.

> **Exercise:** This is the student version. Complete the four tasks (`TASK 1` … `TASK 4`) in the helper-functions cell and finish the Q-learning update; each unfinished task currently raises `NotImplementedError`. Hints give the *formula*, not the code. A fully worked version is available in `Q-Learning_Solution.ipynb`.

## Frozen Lake

Frozen lake is a toy text environment involves crossing a frozen lake from start to goal without falling into any holes by walking over the frozen lake. <br>

We can also set the lake to be **slippery**, so that the agent does not always move in the intended direction. We start with the non-slippery (deterministic) case and then train on the slippery (stochastic) version at the end of the notebook.<br>

You can read more about the environment [here](https://gymnasium.farama.org/environments/toy_text/frozen_lake/).

![Frozen Lake](https://gymnasium.farama.org/_images/frozen_lake.gif)


## OpenAI Gymnasium

[OpenAI Gymnasium](https://gymnasium.farama.org/index.html) is a toolkit for developing and comparing reinforcement learning (RL) algorithms. It consists of a growing suite of environments (from simulated robots to Atari games), and a site for comparing and reproducing results. OpenAI Gymnasium provides a diverse suite of environments that range from easy to difficult and involve many different kinds of data.

Creating and Interacting with gymnasium environments is very simple.

```
import gymnasium as gym
env = gym.make("CartPole-v1")
observation, info = env.reset(seed=42)

for _ in range(1000):
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        observation, info = env.reset()
env.close()
```

Following are the definitions of some common terminologies used.

**Reset:** Resets the environment to an initial state and returns the initial observation. <br>
**Step:** Run one timestep of the environment's dynamics.<br>
**Observation:** The observed state of the environment.<br>
**Action:** An action provided by the agent.<br>
**Reward:** The amount of reward returned as a result of taking the action.<br>
**Terminated:** Whether a terminal state (as defined under the MDP of the task) is reached.<br>
**Truncated:** Whether a truncation condition outside the scope of the MDP is satisfied. Typically a timelimit, but could also be used to indicate agent physically going out of bounds.<br>
**Info:** This contains auxiliary diagnostic information (helpful for debugging, learning, and logging).<br>
**Action Space:** This attribute gives the format of valid actions. It is of datatype Space provided by Gym. For example, if the action space is of type Discrete and gives the value Discrete(2), this means there are two valid discrete actions: 0 & 1.<br>
**Observation:** This attribute gives the format of valid observations. It is of datatype Space provided by Gym. For example, if the observation space is of type Box and the shape of the object is (4,), this denotes a valid observation will be an array of 4 numbers.<br>

Note: Previously, `terminated` and `truncated` used to be merged under one variable `done`. <br>


We will use OpenAI Gymnasium for Frozen Lake environment.

## Q-Learning Algorithm

Q-learning is an algorithm that repeatedly adjusts Q Value to minimize the Bellman error.
$$Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \alpha \underbrace{\left [ r(s,a) + \gamma \max_{a'} Q(s_{t+1},a') - Q(s_t,a_t) \right ]}_{\text{Bellman Error}}$$

The Q-value function at state s and action a, is the expected cumulative reward from taking action a in state s and then following the policy:
$$Q(s,a) = \mathbb{E} \left [ \sum_{t \geq 0} \gamma^t r_t \right ]$$

We learn these Q-values using the Q-learning algorithm.<br>

The discount factor $\gamma$ is the weight for future rewards.<br>


![q_learning.png](https://i.imgur.com/QYmKQxM.png)

[Image Source](https://uoft-csc413.github.io/2022/)

In [ ]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt

In [ ]:
# Create the environment
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array")

### Q-Table

Now, we need to create Q-table. A Q table helps us find the best action for each state. It gives us the Q-value for each state-action pair.<br>

To know how much rows (states) and columns (actions) we need, we need to calculate the action_size and the state_size. OpenAI Gym provides us a way to do that.

In [ ]:
state_size = env.observation_space.n
action_size = env.action_space.n

state_size, action_size

In [ ]:
# Create our Q table with state_size rows and action_size columns (64x4). We can set all values to zero for now.
qtable = np.zeros((state_size, action_size))
print(qtable)

### Exploration vs Exploitation

Notice that Q-learning only learns about the states and actions it visits. What if an optimal state remains unvisited due to not being explored. The agent should sometimes pick suboptimal actions in order to visit new states and actions. <br>

A simple strategy is to use an $\epsilon$-greedy policy. According to this policy, the agent takes a random action with epsilon probability. The value of epsilon is high at the start of training and low towards the end. So, the agent explores more at the start and then exploit the learned policy more at the end.

### Hyperparameters

In [ ]:
# Here, we will specify the hyperparameters

total_episodes = 20000       # Total training episodes
learning_rate = 0.1          # Learning rate
max_steps = 99               # Max steps per episode
gamma = 0.95                 # Discounting rate

# Exploration parameters
epsilon = 1.0                 # Exploration rate
max_epsilon = 1.0             # Exploration probability at start
min_epsilon = 0.01            # Minimum exploration probability
decay_rate = 0.0005           # Exponential decay rate for exploration prob

### Training

In [ ]:
def greedy_action(qtable, state):
    """TASK 1: return the action with the highest Q-value in `state`.
    HINT: look at the row qtable[state, :] and return the INDEX of its largest value."""
    raise NotImplementedError("TASK 1: implement greedy_action")


def epsilon_greedy(qtable, state, epsilon, env):
    """TASK 2: with probability `epsilon` EXPLORE (a random action), otherwise
    EXPLOIT (the greedy action).
    HINT: draw a number with random.uniform(0, 1); env.action_space.sample() returns
    a random action; reuse greedy_action for the exploit branch."""
    raise NotImplementedError("TASK 2: implement epsilon_greedy")


def train_q_learning(env, total_episodes, learning_rate, max_steps, gamma,
                     max_epsilon=1.0, min_epsilon=0.01, decay_rate=0.0005):
    """Tabular Q-learning. Returns the learned Q-table and the per-episode rewards.
    The loop is provided; you complete the learning update in TASK 3."""
    qtable = np.zeros((env.observation_space.n, env.action_space.n))
    epsilon = max_epsilon
    rewards = []

    for episode in range(total_episodes):
        state, _ = env.reset()
        total_rewards = 0

        for step in range(max_steps):
            action = epsilon_greedy(qtable, state, epsilon, env)   # uses your TASK 2
            new_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            # TASK 3: update qtable[state, action] with the Q-learning rule.
            #   new value = old value + learning_rate * (TD_target - old value)
            #   TD_target = reward + gamma * (best Q-value available in new_state)
            #   IMPORTANT: when the episode is `done` there is no next state, so the
            #   bootstrap term (best Q-value in new_state) must contribute 0.
            raise NotImplementedError("TASK 3: implement the Q-learning update")
            # qtable[state, action] = ...

            total_rewards += reward
            state = new_state
            if done:
                break

        # decay epsilon: explore a lot early, exploit more as Q improves (provided)
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
        rewards.append(total_rewards)

    return qtable, rewards


def evaluate_policy(env, qtable, n_episodes=1000, max_steps=99):
    """TASK 4: run the GREEDY policy (no exploration) for n_episodes and return the
    success rate (fraction of episodes that reach the goal).
    HINT: reset the env each episode and step with greedy_action until the episode
    terminates/truncates. On FrozenLake the per-step reward is 1 only when the agent
    reaches the goal, so a successful episode contributes 1 to the count."""
    raise NotImplementedError("TASK 4: implement evaluate_policy")


def plot_rewards(rewards, window=500, title="Training progress"):
    """Plot the moving-average reward. On FrozenLake (reward 1 on success, else 0)
    this moving average is exactly the recent success rate during training."""
    rewards = np.asarray(rewards, dtype=float)
    if len(rewards) >= window:
        ma = np.convolve(rewards, np.ones(window) / window, mode="valid")
    else:
        ma = rewards
    plt.figure(figsize=(12, 4))
    plt.plot(ma)
    plt.xlabel("Episode")
    plt.ylabel(f"Success rate (avg over {window} episodes)")
    plt.title(title)
    plt.ylim(-0.05, 1.05)
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
qtable, rewards = train_q_learning(
    env,
    total_episodes=total_episodes,
    learning_rate=learning_rate,
    max_steps=max_steps,
    gamma=gamma,
    max_epsilon=max_epsilon,
    min_epsilon=min_epsilon,
    decay_rate=decay_rate,
)

plot_rewards(rewards, window=500, title="Q-learning on deterministic FrozenLake")

# Evaluate the GREEDY policy (no exploration) on a fresh environment
eval_env = gym.make("FrozenLake-v1", is_slippery=False)
det_success = evaluate_policy(eval_env, qtable)
print(f"Greedy success rate (deterministic FrozenLake): {det_success:.3f}")

### Visualization

In [ ]:
# Visualization helpers (Gymnasium-native, no dependency on the old `gym` package)
import os
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")  # headless rendering (e.g. Colab)
import imageio.v2 as imageio
from IPython.display import Image, display

os.makedirs("video", exist_ok=True)

In [ ]:
def record_gif(env, qtable, name, max_steps=99, fps=4):
    """Roll out the greedy policy from `qtable` and save the episode as a GIF."""
    frames = []
    state, _ = env.reset()
    for _ in range(max_steps):
        frames.append(env.render())
        action = int(np.argmax(qtable[state, :]))
        state, reward, terminated, truncated, info = env.step(action)
        if terminated or truncated:
            frames.append(env.render())
            break
    env.close()
    path = f"video/{name}.gif"
    imageio.mimsave(path, frames, fps=fps, loop=0)
    return path

def show_gif(name):
    display(Image(filename=f"video/{name}.gif"))

In [ ]:
# Re-create the env with rendering, then watch the greedy policy
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array")
record_gif(env, qtable, "frozenlake")

In [ ]:
show_gif("frozenlake")

## Slippery Frozen Lake (stochastic transitions)

So far the lake was deterministic: the agent always moved in the direction it chose. The real MDP framework allows the environment to be **stochastic** — the next state is drawn from a transition distribution $P(s' \mid s, a)$.

With `is_slippery=True`, the intended action only succeeds with probability $1/3$; with probability $2/3$ the agent slips and moves in one of the two perpendicular directions instead:

- $P(\text{intended direction}) = 1/3$
- $P(\text{each perpendicular direction}) = 1/3$

This is exactly the $P(s' \mid s, a)$ from the MDP definition, and it is why the Bellman update takes an **expectation** over next states. Q-learning still applies unchanged — it learns from sampled transitions — but the problem is harder, so we train for more episodes and let exploration decay more slowly.

In [ ]:
env_slippery = gym.make("FrozenLake-v1", is_slippery=True, render_mode="rgb_array")

In [ ]:
qtable_slippery, rewards_slippery = train_q_learning(
    env_slippery,
    total_episodes=50000,
    learning_rate=0.1,
    max_steps=99,
    gamma=0.99,
    max_epsilon=1.0,
    min_epsilon=0.01,
    decay_rate=0.0001,
)

plot_rewards(rewards_slippery, window=1000, title="Q-learning on slippery FrozenLake")

# Evaluate the greedy policy on the slippery lake and compare with the deterministic case
eval_env_slippery = gym.make("FrozenLake-v1", is_slippery=True)
slip_success = evaluate_policy(eval_env_slippery, qtable_slippery)
print(f"Greedy success rate (slippery FrozenLake):       {slip_success:.3f}")
print(f"Greedy success rate (deterministic FrozenLake):  {det_success:.3f}")

**What to notice:**

- The success rate is well **below 1.0** even after convergence. Because the agent slips, it cannot guarantee reaching the goal — the *optimal* policy here maximizes the *probability* of success, not a guaranteed path.
- The optimal policy often looks counter-intuitive: it steers **away** from holes (e.g. hugging walls) so that an accidental slip is less likely to be fatal.
- Learning is **noisier and slower** than the deterministic case — the same state-action pair can lead to different outcomes, so each Q-value is an average over many sampled transitions.

In [ ]:
env_slippery = gym.make("FrozenLake-v1", is_slippery=True, render_mode="rgb_array")
record_gif(env_slippery, qtable_slippery, "frozenlake_slippery")

In [ ]:
show_gif("frozenlake_slippery")